In [6]:
import os
from dotenv import load_dotenv

from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
GIGA_KEY = os.getenv("GIGA_KEY")
print("Ключ загружен:", bool(GIGA_KEY))

Ключ загружен: True


In [7]:
llm = GigaChat(
    credentials=GIGA_KEY,
    model="GigaChat",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000
)

# Проверка подключения
response = llm.invoke("Привет! Как дела?")
print(response.content)

Привет! Всё отлично, готов общаться и помогать тебе с любыми вопросами. А ты как настроение держишь сегодня?


In [8]:
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.

Текст заявки: {text}

Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.

Количество человек:"""
)

chain = basic_prompt | llm | StrOutputParser()

# Тестирование на примерах из задания
test_texts = [
    "Ищу квартиру для семьи из четырех человек на длительный срок",
    "Нужна студия для проживания одного человека рядом с метро",
    "Требуется двухкомнатная квартира для молодой пары",
    "Снимем жилье для троих студентов на учебный год",
    "Семья с двумя детьми ищет просторную квартиру"
]

for text in test_texts:
    result = chain.invoke({"text": text})
    print(f"Текст: {text}")
    print(f"Результат: {result}")
    print("---")

Текст: Ищу квартиру для семьи из четырех человек на длительный срок
Результат: 4
---
Текст: Нужна студия для проживания одного человека рядом с метро
Результат: 1
---
Текст: Требуется двухкомнатная квартира для молодой пары
Результат: 2
---
Текст: Снимем жилье для троих студентов на учебный год
Результат: 3
---
Текст: Семья с двумя детьми ищет просторную квартиру
Результат: 4
---


In [9]:
import pandas as pd

df = pd.read_csv('rental_10.csv', sep=';')  
print(df.head())
print(f"Всего строк: {len(df)}")
print(f"Столбцы: {df.columns.tolist()}")

                                                text  amount
0  Ищу квартиру для семьи из четырех человек на д...       4
1  Нужна студия для проживания одного человека ря...       1
2  Требуется двухкомнатная квартира для молодой пары       2
3    Снимем жилье для троих студентов на учебный год       3
4      Семья с двумя детьми ищет просторную квартиру       4
Всего строк: 11
Столбцы: ['text', 'amount']


In [10]:
results = []

for _, row in df.iterrows():
    text = row["text"]
    try:
        result = chain.invoke({"text": text})
        results.append(result.strip())
    except Exception as e:
        results.append(f"ERROR: {e}")

df["result"] = results
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")
print("Готово! Сохранено в rental_with_results.csv")
print(df[["text", "amount", "result"]].head(10))

Готово! Сохранено в rental_with_results.csv
                                                text  amount result
0  Ищу квартиру для семьи из четырех человек на д...       4      4
1  Нужна студия для проживания одного человека ря...       1      1
2  Требуется двухкомнатная квартира для молодой пары       2      2
3    Снимем жилье для троих студентов на учебный год       3      3
4      Семья с двумя детьми ищет просторную квартиру       4      4
5  Ищу жилье для одного студента неподалеку от ун...       1      1
6               Пара ищет квартиру с двумя спальнями       2      2
7         Четверо коллег хотят снять квартиру на год       4      4
8  Необходимо жилье для молодой семьи с тремя детьми       5      5
9    Ищу комнату для одного человека в центре города       1      1


In [ ]:
# Смотрим типы данных
print(df.dtypes)

# Преобразуем result из строки в число
df["result_num"] = pd.to_numeric(df["result"], errors='coerce')

# Сравниваем с правильными ответами
correct = (df["result_num"] == df["amount"]).sum()
total = len(df)
errors = total - correct
accuracy = correct / total

print(f"Верных ответов: {correct}")
print(f"Ошибок: {errors}")
print(f"Точность: {accuracy:.1%}")

# Показываем где ошибки
print("\nОшибочные строки:")
print(df[df["result_num"] != df["amount"]][["text", "amount", "result_num"]])